# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmananwar-AI/flyrank-ml-internship-usman/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from google.colab import userdata
import os

# Load Hugging Face token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')


Unit of Analysis: One piece of content on a specific day.

Tables Used: fact_content_daily_performance (mid-panel month).

Time Window: March 1, 2026, to March 31, 2026 (2026-03), reserving June 2026 as a sealed test month.

Target to Predict: Content Drop-off Risk (Proxy: Content had GA4 sessions, but 0 engaged sessions).

Deliberately Excluded: Future interaction metrics recorded after the daily snapshot.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [7]:
import os
import pandas as pd
import numpy as np
from google.colab import userdata
from datasets import load_dataset
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

# ==========================================
# 1. AUTHENTICATION & DATA LOADING
# ==========================================
print("Authenticating and loading data...")
hf_token = userdata.get('HF_TOKEN')

# Load the mid-panel month (March 2026) directly from the parquet partition
dataset = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/*.parquet",
    token=hf_token
)
df = dataset['train'].to_pandas()

# ==========================================
# 2. VERIFICATION QUERIES
# ==========================================
print("\n--- Running Verification Queries ---")

# Query 1: Verify Grain (1 Row = 1 piece of content on a specific day)
total_rows = len(df)
print(f"Query 1 - Grain Check: Total Rows Loaded = {total_rows}")

# Query 2: Row Count and Date Span
min_date = df['report_date'].min()
max_date = df['report_date'].max()
print(f"Query 2 - Slice Stats: Total Rows = {total_rows} | Date Span: {min_date} to {max_date}")

# Query 3: Availability Filter (IS TRUE)
# We only want rows where GA4 data successfully recorded
df_valid = df[df['ga4_data_available'] == True].copy()
surviving_rows = len(df_valid)
print(f"Query 3 - Availability Check: {surviving_rows} rows survive `ga4_data_available IS TRUE` filter.")


# ==========================================
# 3. FEATURE FRAME & LEAKAGE TRAP
# ==========================================
print("\n--- Building Feature Frame ---")

# Define the Target (Proxy for Drop-off)
# A "drop off" here is when a piece of content gets sessions, but 0 engaged sessions.
df_valid = df_valid[df_valid['ga4_sessions'] > 0].copy()
df_valid['content_drop_off_risk'] = (df_valid['ga4_engaged_sessions'] == 0).astype(int)

# Downsample for memory safety in Colab (prevents RAM crashes on millions of rows)
df_sample = df_valid.sample(n=min(20000, len(df_valid)), random_state=42).copy()
df_sample = df_sample.fillna(0)

# The 5 Features
# 1. gsc_impressions: Knowable because search visibility happens before the user arrives.
# 2. gsc_avg_position: Knowable because ranking is established prior to the session.
# 3. sessions_organic: Knowable because the traffic source is logged at arrival.
# 4. sessions_ai: Knowable because the referrer (ChatGPT/Claude) is logged at arrival.
# 5. client_has_gsc: Knowable because client configuration is static metadata.
features = ['gsc_impressions', 'gsc_avg_position', 'sessions_organic', 'sessions_ai', 'client_has_gsc']

X = df_sample[features]
y = df_sample['content_drop_off_risk']

# Train Honest Model
print("\nTraining models...")
model = RandomForestClassifier(random_state=42)
model.fit(X, y)
honest_probs = model.predict_proba(X)[:, 1]
honest_pr_auc = average_precision_score(y, honest_probs)
print(f"Honest Model PR-AUC (5 Features): {honest_pr_auc:.4f}")

# The Trap: Adding a Label-Derived Feature (Data Leakage)
# TRAP: ga4_total_engagement_sec. If engagement seconds = 0, it perfectly correlates with our drop-off target.
X_leaked = df_sample[features + ['ga4_total_engagement_sec']]
model_leaked = RandomForestClassifier(random_state=42)
model_leaked.fit(X_leaked, y)
leaked_probs = model_leaked.predict_proba(X_leaked)[:, 1]
leaked_pr_auc = average_precision_score(y, leaked_probs)

print(f"LEAKED Model PR-AUC (With Trap Feature 'ga4_total_engagement_sec'): {leaked_pr_auc:.4f} <-- Artificial perfect score!")

# Removing the Trap Column
del X_leaked
print("Trap feature removed. Restored honest data contract.")

Authenticating and loading data...

--- Running Verification Queries ---
Query 1 - Grain Check: Total Rows Loaded = 9841378
Query 2 - Slice Stats: Total Rows = 9841378 | Date Span: 2026-03-01 to 2026-03-31
Query 3 - Availability Check: 413966 rows survive `ga4_data_available IS TRUE` filter.

--- Building Feature Frame ---


/tmp/ipykernel_1353/3730576085.py:56: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sample = df_sample.fillna(0)



Training models...
Honest Model PR-AUC (5 Features): 0.9986
LEAKED Model PR-AUC (With Trap Feature 'ga4_total_engagement_sec'): 1.0000 <-- Artificial perfect score!
Trap feature removed. Restored honest data contract.


Limitation: Relying strictly on a single mid-panel month creates a cold-start problem. If a client URL was just published late in the month, we have no historical performance baseline for it compared to URLs that have been active for years.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.